# Direction
I had quite a number of ideas on what might make good functions for fitness, selection, crossing and so on.
Having some trouble in deciding which to pick and how to mix em, I decide to just try all possible combinations

the main thing I want to test by doing this is whenever the Demutate Function as it will be defined later can be a usefull for finding good solutions.

# Problem
The task is to program an evolutionary algorithm which solves the backpack problem

## Problem parameters
- max_weight
- item_weights
- item_values

# Design Structure
## Independent modules
for simplicity sake all functions which are indented to be modular, and thus will follow the following format
```python
def Function(Primary_argument, *TerniaryArguments, **MetaArguments):
    ...
```
to the algorithm itself it will be presented as ` lambda *x: Function(x,**defaultparams)`. the paraeters in `*ListArguments` are intended to be the actual parameters of the algorithm, while `**MetaArguments` should be the specialization of the function.

what follows is the example implementation of the identity function under this format:

In [ ]:
# intended for testing purpuses, does nothing.
def identity(primaryinput = None, *regparams, **params):
    return primaryinput

for simplicity the following function will be used to pass these general functions with the desired metaparametrs applied

In [ ]:
# applies some arguments to a generalized function
def Apply(func, Args):
    return lambda *x: func(*x, **Args)

## Evolution Framework
The evolution symulation framework, termed henceforth evolutor, is defined as a class with the following methods:
- `__init__(*Problem_Spec, /,**Meta_Args)`
    - accepts the problem specification and a full list of the evolution experiment's meta parameters (detailed later)
    - DOES NOT SETUP THE EXPERIMENT.
- `Setup()`
    - generates a random population and sets all relevant variables to their initial states to prepare for a run.
    - intended to permit running several evolution attempts with the same meta-parameters
- `ProcessGeneration(Silent=False)`
    - simulates a single generation with the following order of operations:
        1. Selection
        2. Crossing
        3. Mutation
        4. Demutation
        5. Fitness Evaluation
    - logs relevant generation statistics if `Silent is false` 
        - otherwise rewrites last generations information (usefull if you dont care about the stats and only want a result)
- `Run(Silent=False)`
    - simulates generations until the target generation is reached
    - the silent again determines whenver logging occurs
- `GiveStats()`
    - returns a dictionary with the collected stats of the LAST experiment

### Evolution Metaparameters
The following is exhaustive list of all the meta parameters accepted by the system alongside a brief description of them:
- gen_count
    - Integer
    - for how many generation should the experiment run
- Pop_char
    - 3 Integer Tuple
    - charaterises the size and composition of a population
- fitness_func
    - Func: Individual -> Fitness
    - a function that maps any individual to their fitness value
- populate_func 
    - Func: Integer -> List(Individual)
    - a function that maps any integer value to a generated population list of that size
- selector_func 
    - Func: List(Individual), List(Fitness), Integer -> List(Individual)
    - Performs a selection of some number of individuals from the given population based on their fitnesses
- cross_func
    - Func: Individual, Individual -> Individual
    - accepts a pair of individuals and return a singular individual defined as the cross of the the original two
- mutate_func
    - Func: Individual -> Individual
    - accepts a single individual and performs some mutation upon it
- demutate_func
    - Func: Individual -> Individual
    - accepts a single individual and detemuates it so as to correct some property
        - in this example its goal is to turn an individual which is inadmissable into some nearby "more admissable" individual
        - for example in an overweighted backpack example it could remove a random element.

## Function Descriptions
these are descriptions of the different defined meta-parameter functions accepted by the evolutor and exhaustive lists of defined instances of these functions


### Population Generators
A function that accepts a given value $n$ and returns a population of individuals of that size.


### Fitness Evaluators
A function that accepts an individual and returns a fitness value, depending on the definition of other functions this value may need to be positive.
some preferable properties may include:



### Population Selectors
A function that accepts some population $Pop$ along with its associated fitness values $Fit$ and a value $n$ it returns some multiset of the population such that the size of this set is equal to $n$.


### Individual Crossers
A function that accepts two individuals $a,b$ and returns some individual $c$ derived from the original pair of individuals.

Some preferable properties may include:
- $a=b\Rightarrow a=c=b$
- we can define some space of individuals so that for a line $L$ it holds $a,b\in L\Rightarrow c\in L$
- $\text{Cross}(a,b)=\text{b,a}$

##### General Crossing functions
we can define these more generally as taking some $n$ number of individuals and again returning some $k$ number of resulting new individuals.


### Individual Mutators
A function that accepts an individual and returns an altered version of this individual
##### Induced Mutators
given some original individual $x$, a crossing function $\text{Cross}(a,b)$ and a random individual generator $\text{gen}()$ we can define a mutation function as some binary tree where the leaves are either $x$ or the product of $\text{gen}()$ and every other note with the child nodes $l,r$ represents the individual $\text{Cross}(l,r)$.

an example would be for example the following $\text{Mut}(x):=\text{Cross}(\text{Cross}(\text{gen}(),x),\text{Cross}(x,\text{gen}()))$ for or $\text{Mut}(x):=\text{Cross}(\text{Cross}(x,\text{gen}()),x)$

for a more general multi argument corssing function we can denote higher order $n$-ary trees.

none of the mutators present in this code are of this type as they can often be defined much more efficiently. But this is still a pretty intresting idea which I wanted to note down.


### Individual Demutators
A function that accepts an individual and performs some alteration on it that is guranteed to not decrease its fitness
- an example related to the backpack problem and a fitness that evaluates all unadmissable elements as $0$. would be to remove items from the individual untill they become an admissable solution, this might point to secondary good name and that being Directed Mutation
- as the primary goal is to test what if any effect the application of these function has on the evolutionary algorithm. 